# 4. Agent — cAIuldron v2.0

LangGraph StateGraph with 5 nodes:

```
START → detect_ingredients → estimate_nutrition
         → [rag_search || web_search] (parallel)
         → generate_recipes → generate_images → END
```

In [ ]:
import time
from typing import TypedDict, List, Dict, Optional, Annotated
import operator
import json

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

print('✅ Agent imports OK')

## State Definition

In [ ]:
class RecipeState(TypedDict):
    # Input
    image_bytes:               bytes

    # Stage 1: detect_ingredients
    detected_ingredients:      List[str]
    ingredient_detection_raw:  str

    # Stage 2: estimate_nutrition
    nutrition_per_ingredient:  Dict
    nutrition_summary:         str

    # Stage 3a: rag_search (parallel)
    rag_retrieved_recipes:     List[Dict]
    rag_context_block:         str

    # Stage 3b: web_search (parallel)
    web_search_results:        List[Dict]
    web_context_block:         str

    # Stage 4: generate_recipes
    recipes:                   List[Dict]

    # Stage 5: generate_images (mutates recipes in-place with image_bytes)
    # (no new keys — recipes list updated)

    # Tracking — both use reducers so parallel nodes can write safely
    errors:                    Annotated[List[str], operator.add]
    processing_time_seconds:   Annotated[Dict[str, float], lambda a, b: {**a, **b}]

print('✅ RecipeState defined')

## LLM Client (ChatGroq — traced by LangSmith automatically)

In [ ]:
_chat_llm = ChatGroq(
    model=GROQ_TEXT_MODEL,
    temperature=GROQ_TEMPERATURE,
    api_key=GROQ_API_KEY,
)

print(f'✅ ChatGroq ready: {GROQ_TEXT_MODEL}')

## Node 1: Detect Ingredients

In [ ]:
def detect_ingredients_node(state: RecipeState) -> dict:
    t0 = time.time()
    ingredients, raw = detect_ingredients_from_image(state['image_bytes'])

    result = {
        'detected_ingredients':     ingredients if ingredients else ['unknown ingredient'],
        'ingredient_detection_raw': raw,
        'processing_time_seconds':  {'detect_ingredients': round(time.time() - t0, 2)},
    }
    if not ingredients:
        result['errors'] = ['Vision API returned no ingredients — check image quality']
    return result

print('✅ Node 1: detect_ingredients_node')

## Node 2: Estimate Nutrition

In [ ]:
def estimate_nutrition_node(state: RecipeState) -> dict:
    t0 = time.time()
    per_ing, summary = build_nutrition_summary(state['detected_ingredients'])
    return {
        'nutrition_per_ingredient': per_ing,
        'nutrition_summary':        summary,
        'processing_time_seconds':  {'estimate_nutrition': round(time.time() - t0, 2)},
    }

print('✅ Node 2: estimate_nutrition_node')

## Node 3a: RAG Search (parallel)

In [ ]:
def rag_search_node(state: RecipeState) -> dict:
    t0 = time.time()
    retrieved, context = retrieve_similar_recipes(state['detected_ingredients'])
    return {
        'rag_retrieved_recipes': retrieved,
        'rag_context_block':     context,
        'processing_time_seconds': {'rag_search': round(time.time() - t0, 2)},
    }

print('✅ Node 3a: rag_search_node')

## Node 3b: Web Search (parallel)

In [ ]:
def web_search_node(state: RecipeState) -> dict:
    t0 = time.time()
    try:
        results, context = search_recipe_inspiration(state['detected_ingredients'])
    except Exception as e:
        return {
            'web_search_results': [],
            'web_context_block':  'Web search unavailable.',
            'errors': [f'Web search failed: {e}'],
            'processing_time_seconds': {'web_search': round(time.time() - t0, 2)},
        }
    return {
        'web_search_results': results,
        'web_context_block':  context,
        'processing_time_seconds': {'web_search': round(time.time() - t0, 2)},
    }

print('✅ Node 3b: web_search_node')

## Node 4: Generate Recipes (Groq 70B with RAG + Web context)

In [ ]:
_RECIPE_SYSTEM = """You are a professional chef and culinary writer with Michelin-star experience.
Generate exactly 5 complete, restaurant-quality recipes as a JSON object with key "recipes" containing an array.
Each recipe MUST follow this exact schema:
{
  "title": "string",
  "cuisine": "string",
  "difficulty": "easy|medium|hard",
  "prep_time_minutes": integer,
  "cook_time_minutes": integer,
  "total_time_minutes": integer,
  "servings": integer,
  "ingredients": ["precise amount + ingredient + preparation note", ...],
  "instructions": ["Step 1: ...", "Step 2: ...", ...]
}
Rules:
- Return ONLY valid JSON, no markdown fences, no commentary
- Each recipe must use the detected ingredients as PRIMARY ingredients
- Make each recipe genuinely different in technique, cuisine, and flavour profile
- ingredients list: minimum 10 items with precise measurements (e.g. "2 tbsp extra-virgin olive oil", "1 tsp freshly ground black pepper")
- instructions list: minimum 8 detailed steps — include prep techniques, temperatures, timing cues, visual doneness cues, and plating tips
- Instructions must read like a cookbook: vivid, specific, professional (e.g. "Sear over high heat until a golden-brown crust forms, about 3–4 minutes per side")
- Cooking times must be realistic (total = prep + cook)
- Include at least one flavour-building step (e.g. deglazing, blooming spices, building a sauce)"""


def generate_recipes_node(state: RecipeState) -> dict:
    t0 = time.time()

    ingredients_str = ', '.join(state['detected_ingredients'])
    cuisine_list = '\n'.join(
        f"  Recipe {i+1}: {c['cuisine']} style, {c['difficulty']} difficulty"
        for i, c in enumerate(CUISINE_ROTATION[:NUM_RECIPES])
    )

    user_prompt = (
        f"Detected ingredients: {ingredients_str}\n\n"
        f"{state.get('rag_context_block', '')}\n\n"
        f"{state.get('web_context_block', '')}\n\n"
        f"Nutrition context:\n{state.get('nutrition_summary', '')}\n\n"
        f"Generate these 5 recipe styles:\n{cuisine_list}\n\n"
        f"Each recipe must have at least 10 ingredients and 8 detailed cooking steps.\n"
        f'Return a JSON object with key "recipes" containing an array of 5 complete recipes.'
    )

    try:
        response = _chat_llm.invoke([
            SystemMessage(content=_RECIPE_SYSTEM),
            HumanMessage(content=user_prompt),
        ])
        raw_json = response.content.strip()

        # Strip markdown fences if present
        import re
        raw_json = re.sub(r'^```(?:json)?\s*', '', raw_json, flags=re.MULTILINE)
        raw_json = re.sub(r'```\s*$', '', raw_json, flags=re.MULTILINE).strip()

        parsed = json.loads(raw_json)
        recipes_raw = parsed.get('recipes', parsed) if isinstance(parsed, dict) else parsed

        recipes = []
        rag_titles = [r['title'] for r in state.get('rag_retrieved_recipes', [])]

        for i, r in enumerate(recipes_raw[:NUM_RECIPES]):
            cfg = CUISINE_ROTATION[i] if i < len(CUISINE_ROTATION) else CUISINE_ROTATION[0]
            recipes.append({
                'title':               r.get('title', f'Recipe {i+1}'),
                'cuisine':             r.get('cuisine', cfg['cuisine']),
                'difficulty':          r.get('difficulty', cfg['difficulty']),
                'prep_time_minutes':   int(r.get('prep_time_minutes', 15) or 15),
                'cook_time_minutes':   int(r.get('cook_time_minutes', 25) or 25),
                'total_time_minutes':  int(r.get('total_time_minutes', 40) or 40),
                'servings':            int(r.get('servings', 4) or 4),
                'ingredients':         r.get('ingredients', [ingredients_str]),
                'instructions':        r.get('instructions', []),
                'image_bytes':         None,   # filled by generate_images_node
                'rag_source_titles':   rag_titles,
            })

        return {
            'recipes': recipes,
            'processing_time_seconds': {'generate_recipes': round(time.time() - t0, 2)},
        }

    except Exception as e:
        return {
            'recipes': [],
            'errors':  [f'Recipe generation failed: {e}'],
            'processing_time_seconds': {'generate_recipes': round(time.time() - t0, 2)},
        }

print('✅ Node 4: generate_recipes_node')

## Node 5: Generate Dish Images (FLUX.1-schnell via HF Inference API)

In [ ]:
def generate_images_node(state: RecipeState) -> dict:
    t0 = time.time()
    recipes = state.get('recipes', [])

    if not recipes:
        return {'processing_time_seconds': {'generate_images': 0}}

    try:
        print(f'  Generating {len(recipes)} dish images via FLUX.1-schnell...')
        image_bytes_list = generate_images_sync(recipes)
    except Exception as e:
        return {
            'errors': [f'Image generation failed: {e}'],
            'processing_time_seconds': {'generate_images': round(time.time() - t0, 2)},
        }

    updated_recipes = []
    for i, recipe in enumerate(recipes):
        img = image_bytes_list[i] if i < len(image_bytes_list) else None
        updated_recipes.append({**recipe, 'image_bytes': img})
        status = '✅' if img else '❌'
        print(f'  {status} {recipe["title"]}')

    return {
        'recipes': updated_recipes,
        'processing_time_seconds': {'generate_images': round(time.time() - t0, 2)},
    }

print('✅ Node 5: generate_images_node')

## Compile LangGraph StateGraph

In [ ]:
def build_recipe_graph():
    graph = StateGraph(RecipeState)

    # Register all nodes
    graph.add_node('detect_ingredients',  detect_ingredients_node)
    graph.add_node('estimate_nutrition',  estimate_nutrition_node)
    graph.add_node('rag_search',          rag_search_node)
    graph.add_node('web_search',          web_search_node)
    graph.add_node('generate_recipes',    generate_recipes_node)
    graph.add_node('generate_images',     generate_images_node)

    # Linear edges: start → detect → nutrition
    graph.add_edge(START,                'detect_ingredients')
    graph.add_edge('detect_ingredients', 'estimate_nutrition')

    # Parallel fork: nutrition → rag_search AND web_search simultaneously
    graph.add_edge('estimate_nutrition', 'rag_search')
    graph.add_edge('estimate_nutrition', 'web_search')

    # Parallel join: both must complete before generate_recipes
    graph.add_edge('rag_search',  'generate_recipes')
    graph.add_edge('web_search',  'generate_recipes')

    # Continue: generate_recipes → images → END
    graph.add_edge('generate_recipes', 'generate_images')
    graph.add_edge('generate_images',  END)

    return graph.compile()


recipe_graph = build_recipe_graph()
print('✅ LangGraph compiled')
print('   Nodes: detect_ingredients → estimate_nutrition')
print('          → [rag_search || web_search] → generate_recipes → generate_images → END')